In [42]:
import sys
from pathlib import Path

import pandas as pd
import geopandas as gpd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import PROCESSED_DIR
from src.utils import preview

In [2]:
nyc_model_features_gdf = gpd.read_file(PROCESSED_DIR / "nyc_model_features.geojson")
preview(nyc_model_features_gdf, "NYC Model Features")
nyc_model_features_gdf.head()

NYC Model Features: 206 rows x 14 cols
  zip_code   boroname  total_population  median_household_income  \
0    10028  Manhattan             45679                 168125.0   
1    11427     Queens             25124                  92129.0   
2    11697     Queens              4123                 134844.0   
3    11004     Queens             14296                 109865.0   
4    10282  Manhattan              5960                 250001.0   

   median_household_income_filled  nearest_tj_distance_miles  \
0                        168125.0                   1.231225   
1                         92129.0                   5.405433   
2                        134844.0                  10.037851   
3                        109865.0                   7.351747   
4                        250001.0                   0.836176   

   subway_station_count  area_sq_miles  population_density  income_percentile  \
0                   2.0       0.315510       144778.293180           0.941748   
1    

,zip_code,boroname,total_population,median_household_income,median_household_income_filled,nearest_tj_distance_miles,subway_station_count,area_sq_miles,population_density,income_percentile,transit_access_tier,is_tj_gap,log_population_density,geometry
0,10028,Manhattan,45679,168125.0,168125.0,1.231225,2.0,0.315510,144778.293180,0.941748,medium,0,11.882966,"POLYGON ((994561.592 222839.437, 994690.277 22..."
1,11427,Queens,25124,92129.0,92129.0,5.405433,0.0,1.568670,16016.113999,0.631068,low,1,9.681413,"POLYGON ((1049040.311 204011.201, 1049258.374 ..."
2,11697,Queens,4123,134844.0,134844.0,10.037851,0.0,2.047392,2013.781402,0.868932,low,1,7.608266,"POLYGON ((1000721.481 136681.744, 1000888.65 1..."
3,11004,Queens,14296,109865.0,109865.0,7.351747,0.0,0.951062,15031.623052,0.771845,low,1,9.617978,"POLYGON ((1061069.68 214228.676, 1061117.367 2..."
4,10282,Manhattan,5960,250001.0,250001.0,0.836176,0.0,0.068667,86795.712887,0.995146,low,0,11.371324,"POLYGON ((979421.96 199752.189, 979617.937 201..."


In [3]:
print("shape:", nyc_model_features_gdf.shape)
print("duplicate zip_code:", nyc_model_features_gdf["zip_code"].duplicated().sum())
print("missing log_population_density:", nyc_model_features_gdf["log_population_density"].isna().sum())
print("missing income_percentile:", nyc_model_features_gdf["income_percentile"].isna().sum())
print("missing nearest_tj_distance_miles:", nyc_model_features_gdf["nearest_tj_distance_miles"].isna().sum())
print("missing subway_station_count:", nyc_model_features_gdf["subway_station_count"].isna().sum())

shape: (206, 14)
duplicate zip_code: 0
missing log_population_density: 0
missing income_percentile: 0
missing nearest_tj_distance_miles: 0
missing subway_station_count: 0


In [4]:
score_df = nyc_model_features_gdf.copy()

score_df["density_score"] = (
    (score_df["log_population_density"] - score_df["log_population_density"].min()) /
    (score_df["log_population_density"].max() - score_df["log_population_density"].min())
)

score_df["income_score"] = (
    (score_df["income_percentile"] - score_df["income_percentile"].min()) /
    (score_df["income_percentile"].max() - score_df["income_percentile"].min())
)

score_df["tj_gap_score"] = (
    (score_df["nearest_tj_distance_miles"] - score_df["nearest_tj_distance_miles"].min()) /
    (score_df["nearest_tj_distance_miles"].max() - score_df["nearest_tj_distance_miles"].min())
)

score_df["transit_score"] = (
    (score_df["subway_station_count"] - score_df["subway_station_count"].min()) /
    (score_df["subway_station_count"].max() - score_df["subway_station_count"].min())
)

In [5]:
score_df[[
    "density_score",
    "income_score",
    "tj_gap_score",
    "transit_score"
]].describe()

,density_score,income_score,tj_gap_score,transit_score
count,206.000000,206.000000,206.000000,206.000000
mean,0.751057,0.502451,0.282832,0.182972
std,0.306994,0.291760,0.229322,0.201915
min,0.000000,0.000000,0.000000,0.000000
25%,0.794344,0.251225,0.082466,0.000000
50%,0.863939,0.502451,0.237694,0.153846
75%,0.918671,0.753676,0.427790,0.307692
max,1.000000,1.000000,1.000000,1.000000


In [6]:
score_df["site_selection_score"] = (
    0.35 * score_df["density_score"] +
    0.25 * score_df["income_score"] +
    0.25 * score_df["tj_gap_score"] +
    0.15 * score_df["transit_score"]
)

In [7]:
score_df = score_df.sort_values("site_selection_score", ascending=False).copy()
score_df["rank"] = range(1, len(score_df) + 1)

score_df[[
    "rank",
    "zip_code",
    "boroname",
    "site_selection_score",
    "density_score",
    "income_score",
    "tj_gap_score",
    "transit_score"
]].head(15)

,rank,zip_code,boroname,site_selection_score,density_score,income_score,tj_gap_score,transit_score
156,1,11694,Queens,0.722937,0.784588,0.720588,0.934277,0.230769
14,2,11201,Brooklyn,0.700947,0.902663,0.950980,0.035235,0.923077
2,3,11697,Queens,0.690057,0.634056,0.872549,1.000000,0.000000
136,4,11215,Brooklyn,0.670269,0.864694,0.960784,0.186642,0.538462
60,5,10007,Manhattan,0.666021,0.899665,1.000000,0.081476,0.538462
54,6,11217,Brooklyn,0.662574,0.913357,0.926471,0.075893,0.615385
3,7,11004,Queens,0.657162,0.801542,0.774510,0.731981,0.000000
118,8,10005,Manhattan,0.656749,0.979238,0.985294,0.132308,0.230769
86,9,11422,Queens,0.651821,0.803416,0.759804,0.722697,0.000000
195,10,10011,Manhattan,0.651820,0.936823,0.892157,0.034338,0.615385


In [8]:
score_df[[
    "rank",
    "zip_code",
    "boroname",
    "total_population",
    "population_density",
    "median_household_income_filled",
    "nearest_tj_distance_miles",
    "subway_station_count",
    "transit_access_tier",
    "is_tj_gap",
    "site_selection_score"
]].head(15)

,rank,zip_code,boroname,total_population,population_density,median_household_income_filled,nearest_tj_distance_miles,subway_station_count,transit_access_tier,is_tj_gap,site_selection_score
156,1,11694,Queens,21430,12264.435966,103792.0,9.379168,3.0,medium,1,0.722937
14,2,11201,Brooklyn,69251,50581.735173,169285.0,0.368924,12.0,high,0,0.700947
2,3,11697,Queens,4123,2013.781402,134844.0,10.037851,0.0,low,1,0.690057
136,4,11215,Brooklyn,70922,32071.912640,180773.0,1.886329,7.0,high,0,0.670269
60,5,10007,Manhattan,7802,48794.462089,250001.0,0.832355,7.0,high,0,0.666021
54,6,11217,Brooklyn,42569,57508.000862,158314.0,0.776398,8.0,high,0,0.662574
3,7,11004,Queens,14296,15031.623052,109865.0,7.351747,0.0,low,1,0.657162
118,8,10005,Manhattan,9238,126779.893387,211810.0,1.341795,3.0,medium,0,0.656749
86,9,11422,Queens,33085,15373.608914,108519.0,7.258699,0.0,low,1,0.651821
195,10,10011,Manhattan,49344,76210.777774,146571.0,0.359934,8.0,high,0,0.651820


In [9]:
score_df.groupby("boroname")["site_selection_score"].agg(["count", "mean", "max"]).sort_values("mean", ascending=False)

,count,mean,max
boroname,,,
Brooklyn,38,0.552869,0.700947
Queens,64,0.515952,0.722937
Bronx,24,0.496376,0.646338
Staten Island,13,0.480201,0.563311
Manhattan,67,0.418829,0.666021


In [10]:
score_df.groupby("boroname").head(3)[[
    "rank",
    "zip_code",
    "boroname",
    "site_selection_score"
]].sort_values(["boroname", "site_selection_score"], ascending=[True, False])

,rank,zip_code,boroname,site_selection_score
189,12,10471,Bronx,0.646338
121,41,10466,Bronx,0.599851
13,49,10465,Bronx,0.591940
14,2,11201,Brooklyn,0.700947
136,4,11215,Brooklyn,0.670269
54,6,11217,Brooklyn,0.662574
60,5,10007,Manhattan,0.666021
118,8,10005,Manhattan,0.656749
195,10,10011,Manhattan,0.651820
156,1,11694,Queens,0.722937


In [11]:
filtered_score_df = nyc_model_features_gdf.copy()

filtered_score_df = filtered_score_df[
    (filtered_score_df["total_population"] >= 10000) &
    (filtered_score_df["subway_station_count"] >= 1)
].copy()

print("Filtered shape:", filtered_score_df.shape)
print("Remaining duplicate zip_code:", filtered_score_df["zip_code"].duplicated().sum())

Filtered shape: (123, 14)
Remaining duplicate zip_code: 0


In [12]:
filtered_score_df["density_score"] = (
    (filtered_score_df["log_population_density"] - filtered_score_df["log_population_density"].min()) /
    (filtered_score_df["log_population_density"].max() - filtered_score_df["log_population_density"].min())
)

filtered_score_df["income_score"] = (
    (filtered_score_df["income_percentile"] - filtered_score_df["income_percentile"].min()) /
    (filtered_score_df["income_percentile"].max() - filtered_score_df["income_percentile"].min())
)

filtered_score_df["tj_gap_score"] = (
    (filtered_score_df["nearest_tj_distance_miles"] - filtered_score_df["nearest_tj_distance_miles"].min()) /
    (filtered_score_df["nearest_tj_distance_miles"].max() - filtered_score_df["nearest_tj_distance_miles"].min())
)

filtered_score_df["transit_score"] = (
    (filtered_score_df["subway_station_count"] - filtered_score_df["subway_station_count"].min()) /
    (filtered_score_df["subway_station_count"].max() - filtered_score_df["subway_station_count"].min())
)

In [13]:
filtered_score_df["site_selection_score"] = (
    0.40 * filtered_score_df["density_score"] +
    0.25 * filtered_score_df["income_score"] +
    0.20 * filtered_score_df["tj_gap_score"] +
    0.15 * filtered_score_df["transit_score"]
)

In [14]:
filtered_score_df = filtered_score_df.sort_values("site_selection_score", ascending=False).copy()
filtered_score_df["rank"] = range(1, len(filtered_score_df) + 1)

In [15]:
filtered_score_df[[
    "rank",
    "zip_code",
    "boroname",
    "total_population",
    "population_density",
    "median_household_income_filled",
    "nearest_tj_distance_miles",
    "subway_station_count",
    "transit_access_tier",
    "site_selection_score"
]].head(15)

,rank,zip_code,boroname,total_population,population_density,median_household_income_filled,nearest_tj_distance_miles,subway_station_count,transit_access_tier,site_selection_score
0,1,10028,Manhattan,45679,144778.293180,168125.0,1.231225,2.0,medium,0.683385
14,2,11201,Brooklyn,69251,50581.735173,169285.0,0.368924,12.0,high,0.668030
75,3,10003,Manhattan,53825,94148.165059,153750.0,0.125825,7.0,high,0.661704
137,4,10128,Manhattan,59000,130083.869871,148705.0,1.189514,2.0,medium,0.657265
195,5,10011,Manhattan,49344,76210.777774,146571.0,0.359934,8.0,high,0.650634
186,6,10021,Manhattan,39479,110248.904070,156712.0,0.699405,3.0,medium,0.644990
54,7,11217,Brooklyn,42569,57508.000862,158314.0,0.776398,8.0,high,0.635424
71,8,10019,Manhattan,44276,66503.102243,121835.0,0.894757,8.0,high,0.629597
67,9,10025,Manhattan,93223,104881.070194,109195.0,0.511775,5.0,high,0.622063
203,10,10022,Manhattan,34340,77253.595843,171038.0,0.354133,4.0,medium,0.618600


In [16]:
filtered_score_df.groupby("boroname")["site_selection_score"].agg(
    ["count", "mean", "max"]
).sort_values("mean", ascending=False)

,count,mean,max
boroname,,,
Manhattan,33,0.546567,0.683385
Brooklyn,34,0.489074,0.668030
Queens,28,0.422557,0.522347
Bronx,20,0.421476,0.498049
Staten Island,8,0.324964,0.394806


In [17]:
score_df = filtered_score_df.copy()

In [18]:
final_score_df = filtered_score_df[
    filtered_score_df["nearest_tj_distance_miles"] >= 1.0
].copy()

print("Final filtered shape:", final_score_df.shape)
print("Remaining duplicate zip_code:", final_score_df["zip_code"].duplicated().sum())

Final filtered shape: (93, 20)
Remaining duplicate zip_code: 0


In [19]:
final_score_df = final_score_df.sort_values("site_selection_score", ascending=False).copy()
final_score_df["rank"] = range(1, len(final_score_df) + 1)

In [20]:
final_score_df[[
    "rank",
    "zip_code",
    "boroname",
    "total_population",
    "population_density",
    "median_household_income_filled",
    "nearest_tj_distance_miles",
    "subway_station_count",
    "transit_access_tier",
    "site_selection_score"
]].head(15)

,rank,zip_code,boroname,total_population,population_density,median_household_income_filled,nearest_tj_distance_miles,subway_station_count,transit_access_tier,site_selection_score
0,1,10028,Manhattan,45679,144778.293180,168125.0,1.231225,2.0,medium,0.683385
137,2,10128,Manhattan,59000,130083.869871,148705.0,1.189514,2.0,medium,0.657265
136,3,11215,Brooklyn,70922,32071.912640,180773.0,1.886329,7.0,high,0.587102
143,4,11226,Brooklyn,100022,76785.420417,81084.0,3.527844,7.0,high,0.563279
153,5,10036,Manhattan,30589,68396.193634,100225.0,1.232675,4.0,medium,0.555840
32,6,11238,Brooklyn,58462,30184.509861,132957.0,1.774699,7.0,high,0.552231
177,7,11218,Brooklyn,71313,52361.234717,95916.0,3.315765,3.0,medium,0.547652
9,8,11225,Brooklyn,56544,64633.901445,85201.0,2.597065,7.0,high,0.537174
48,9,11211,Brooklyn,65691,43923.215722,105183.0,1.062157,6.0,high,0.535463
193,10,11209,Brooklyn,70756,33481.007330,92656.0,5.066959,3.0,medium,0.527590


In [21]:
final_score_df.groupby("boroname")["site_selection_score"].agg(
    ["count", "mean", "max"]
).sort_values("mean", ascending=False)

,count,mean,max
boroname,,,
Manhattan,10,0.501123,0.683385
Brooklyn,31,0.480698,0.587102
Bronx,20,0.421476,0.498049
Queens,25,0.419856,0.522347
Staten Island,7,0.334900,0.394806


In [22]:
score_df = final_score_df.copy()

In [23]:
shortlist_df = final_score_df[[
    "rank",
    "zip_code",
    "boroname",
    "total_population",
    "population_density",
    "median_household_income_filled",
    "nearest_tj_distance_miles",
    "subway_station_count",
    "transit_access_tier",
    "site_selection_score"
]].head(8).copy()

shortlist_df

,rank,zip_code,boroname,total_population,population_density,median_household_income_filled,nearest_tj_distance_miles,subway_station_count,transit_access_tier,site_selection_score
0,1,10028,Manhattan,45679,144778.293180,168125.0,1.231225,2.0,medium,0.683385
137,2,10128,Manhattan,59000,130083.869871,148705.0,1.189514,2.0,medium,0.657265
136,3,11215,Brooklyn,70922,32071.912640,180773.0,1.886329,7.0,high,0.587102
143,4,11226,Brooklyn,100022,76785.420417,81084.0,3.527844,7.0,high,0.563279
153,5,10036,Manhattan,30589,68396.193634,100225.0,1.232675,4.0,medium,0.555840
32,6,11238,Brooklyn,58462,30184.509861,132957.0,1.774699,7.0,high,0.552231
177,7,11218,Brooklyn,71313,52361.234717,95916.0,3.315765,3.0,medium,0.547652
9,8,11225,Brooklyn,56544,64633.901445,85201.0,2.597065,7.0,high,0.537174


In [24]:
score_df.to_file(
    PROCESSED_DIR / "nyc_scored_candidates.geojson",
    driver="GeoJSON"
)

score_df.drop(columns=["geometry"], errors="ignore").to_csv(
    PROCESSED_DIR / "nyc_scored_candidates.csv",
    index=False
)

In [25]:
competitor_gdf = gpd.read_file(PROCESSED_DIR / "competitors_nyc_geocoded.geojson")
competitor_gdf.head()

,store_name,address,borough,zip_code,unnamed_5,brand,full_address,location,latitude,longitude,geometry
0,Upper East Side,1551 3rd Ave,Manhattan,10128,None,Whole Foods,"1551 3rd Ave, Manhattan, NY 10128","Whole Foods Market, 1551, 3rd Avenue, Carnegie...",40.779697,-73.952949,POINT (-73.95295 40.7797)
1,Lenox Hill,1175 3rd Ave,Manhattan,10065,None,Whole Foods,"1175 3rd Ave, Manhattan, NY 10065","1175, 3rd Avenue, Lenox Hill, Manhattan Commun...",40.767301,-73.962166,POINT (-73.96217 40.7673)
2,Midtown East,226 E 57th St,Manhattan,10022,None,Whole Foods,"226 E 57th St, Manhattan, NY 10022","Whole Foods Market, 226, East 57th Street, Sut...",40.759421,-73.966035,POINT (-73.96603 40.75942)
3,Bryant Park,1095 6th Ave,Manhattan,10036,None,Whole Foods,"1095 6th Ave, Manhattan, NY 10036","Salesforce Tower, 1095, 6th Avenue, Midtown, M...",40.754649,-73.984811,POINT (-73.98481 40.75465)
4,Bowery,95 E Houston St,Manhattan,10002,None,Whole Foods,"95 E Houston St, Manhattan, NY 10002","Whole Foods Market, 95, East Houston Street, M...",40.723848,-73.992371,POINT (-73.99237 40.72385)


In [26]:
top10_refine_gdf = score_df.head(10).copy()
top10_refine_gdf[[
    "rank",
    "zip_code",
    "boroname",
    "site_selection_score"
]]

,rank,zip_code,boroname,site_selection_score
0,1,10028,Manhattan,0.683385
137,2,10128,Manhattan,0.657265
136,3,11215,Brooklyn,0.587102
143,4,11226,Brooklyn,0.563279
153,5,10036,Manhattan,0.555840
32,6,11238,Brooklyn,0.552231
177,7,11218,Brooklyn,0.547652
9,8,11225,Brooklyn,0.537174
48,9,11211,Brooklyn,0.535463
193,10,11209,Brooklyn,0.527590


In [27]:
top10_refine_proj = top10_refine_gdf.to_crs(epsg=2263).copy()
competitor_proj = competitor_gdf.to_crs(epsg=2263).copy()

top10_centroids = top10_refine_proj.geometry.centroid

In [28]:
top10_refine_proj["nearest_competitor_ft"] = top10_centroids.apply(
    lambda pt: competitor_proj.geometry.distance(pt).min()
)

top10_refine_proj["nearest_competitor_miles"] = (
    top10_refine_proj["nearest_competitor_ft"] / 5280
)

In [29]:
def count_competitors_within_radius(point, competitor_geoms, radius_ft=5280):
    return (competitor_geoms.distance(point) <= radius_ft).sum()

top10_refine_proj["competitor_count_1mi"] = top10_centroids.apply(
    lambda pt: count_competitors_within_radius(pt, competitor_proj.geometry, radius_ft=5280)
)

In [30]:
top10_refine_proj[[
    "rank",
    "zip_code",
    "nearest_competitor_miles",
    "competitor_count_1mi"
]]

,rank,zip_code,nearest_competitor_miles,competitor_count_1mi
0,1,10028,0.225418,2
137,2,10128,0.190426,1
136,3,11215,0.855196,1
143,4,11226,2.589715,0
153,5,10036,0.469997,2
32,6,11238,1.207081,0
177,7,11218,2.284980,0
9,8,11225,2.005769,0
48,9,11211,0.276560,2
193,10,11209,4.258736,0


In [31]:
rent_df = pd.read_csv(PROCESSED_DIR / "commercial_rent_top10_clean.csv")
rent_df["zip_code"] = rent_df["zip_code"].astype(str).str.zfill(5)

rent_df[[
    "zip_code",
    "avg_asking_rent_psf"
]]

,zip_code,avg_asking_rent_psf
0,10028,259
1,10128,297
2,11215,141
3,11226,105
4,10036,583
5,11238,105
6,11218,104
7,11225,105
8,11211,250
9,11209,90


In [32]:
top10_refine_df = top10_refine_proj.merge(
    rent_df[["zip_code", "avg_asking_rent_psf", "retail_corridor_proxy"]],
    on="zip_code",
    how="left"
)

In [33]:
print("shape:", top10_refine_df.shape)
print("missing rent:", top10_refine_df["avg_asking_rent_psf"].isna().sum())

top10_refine_df[[
    "rank",
    "zip_code",
    "avg_asking_rent_psf",
    "retail_corridor_proxy"
]]

shape: (10, 25)
missing rent: 0


,rank,zip_code,avg_asking_rent_psf,retail_corridor_proxy
0,1,10028,259,Third Avenue (60th Street - 72nd Street)
1,2,10128,297,East 86th Street (Lexington Avenue - Second Av...
2,3,11215,141,Seventh Ave (Union St - Ninth Street)
3,4,11226,105,Flatbush Ave (Fifth Ave - Grand Army Plaza)
4,5,10036,583,Fifth Avenue (42nd Street - 49th Street)
5,6,11238,105,Flatbush Ave (Fifth Ave - Grand Army Plaza)
6,7,11218,104,Fifth Ave (Union St - Ninth Street)
7,8,11225,105,Flatbush Ave (Fifth Ave - Grand Army Plaza)
8,9,11211,250,Bedford Ave (Grand St - North Eighth Street)
9,10,11209,90,86th Street (4th Ave - Fort Hamilton Pkwy)


In [34]:
top10_refine_df["rent_score"] = 1 - (
    (top10_refine_df["avg_asking_rent_psf"] - top10_refine_df["avg_asking_rent_psf"].min()) /
    (top10_refine_df["avg_asking_rent_psf"].max() - top10_refine_df["avg_asking_rent_psf"].min())
)

In [35]:
top10_refine_df["competition_score"] = 1 - (
    (top10_refine_df["competitor_count_1mi"] - top10_refine_df["competitor_count_1mi"].min()) /
    (top10_refine_df["competitor_count_1mi"].max() - top10_refine_df["competitor_count_1mi"].min())
)

In [36]:
top10_refine_df["tj_customer_fit_score"] = (
    0.50 * top10_refine_df["density_score"] +
    0.35 * top10_refine_df["income_score"] +
    0.15 * top10_refine_df["transit_score"]
)

In [53]:
top10_refine_df["final_refinement_score"] = (
    0.60 * top10_refine_df["site_selection_score"] +
    0.20 * top10_refine_df["tj_customer_fit_score"] +
    0.10 * top10_refine_df["competition_score"] +
    0.10 * top10_refine_df["rent_score"]
)

In [54]:
top10_refine_df = top10_refine_df.sort_values(
    "final_refinement_score",
    ascending=False
).copy()

top10_refine_df["refined_rank"] = range(1, len(top10_refine_df) + 1)

In [57]:
top10_refine_df[[
    "refined_rank",
    "rank",
    "zip_code",
    "boroname",
    "site_selection_score",
    "tj_customer_fit_score",
    "competitor_count_1mi",
    "nearest_competitor_miles",
    "avg_asking_rent_psf",
    "rent_score",
    "final_refinement_score"
]]

,refined_rank,rank,zip_code,boroname,site_selection_score,tj_customer_fit_score,competitor_count_1mi,nearest_competitor_miles,avg_asking_rent_psf,rent_score,final_refinement_score
1,1,2,10128,Manhattan,0.657265,0.821953,1,0.190426,297,0.580122,0.666762
5,2,6,11238,Brooklyn,0.552231,0.658079,0,1.207081,105,0.969574,0.659912
3,3,4,11226,Brooklyn,0.563279,0.604713,0,2.589715,105,0.969574,0.655868
6,4,7,11218,Brooklyn,0.547652,0.615904,0,2.284980,104,0.971602,0.648932
0,5,1,10028,Manhattan,0.683385,0.855393,2,0.225418,259,0.657201,0.646830
7,6,8,11225,Brooklyn,0.537174,0.599026,0,2.005769,105,0.969574,0.639067
2,7,3,11215,Brooklyn,0.587102,0.702494,1,0.855196,141,0.896552,0.632415
9,8,10,11209,Brooklyn,0.527590,0.543308,0,4.258736,90,1.000000,0.625215
8,9,9,11211,Brooklyn,0.535463,0.654320,2,0.276560,250,0.675456,0.519688
4,10,5,10036,Manhattan,0.555840,0.679964,2,0.469997,583,0.000000,0.469497


In [58]:
top10_refine_df.drop(columns=["geometry"], errors="ignore").to_csv(
    PROCESSED_DIR / "top10_refinement_table.csv",
    index=False
)

top10_refine_df.to_file(
    PROCESSED_DIR / "top10_refinement_table.geojson",
    driver="GeoJSON"
)

# Summary
1. Built the initial citywide screening model using normalized versions of density, income, Trader Joe’s whitespace, and transit access.
2. Applied viability filters to remove unrealistic candidates, then reweighted the core model to emphasize density slightly more among viable ZIPs.
3. Ranked candidates citywide and generated the core shortlist of top ZIP codes based on the final screened site_selection_score.
4. Added the finalist-stage refinement overlay for the top 10 using competition and rent proxies, then calculated refinement metrics and final ranks for the recommendation stage.